# Launch TESIS simulations from VS Code

This notebook is a launcher for the **TESIS** stochastic pump-and-treat
optimization code. It is designed for the VS Code Jupyter experience
(select a Python kernel with `numpy` + `scipy` installed; add `torch` only
if you want the batched GPU backend).

**What you can do here**

* Run the self-contained 3D simulation `full3D_colab_run.py` **locally**.
* Offload the same simulation to a **Google Colab** GPU/CPU VM via
  `colab_runner.py`.
* Send the GPU-sized `paperGPU` preset to Colab in one command via
  `SCRIPTS/run_papergpu_colab.py`.
* Browse the run folders (`metrics.json` + PNG figures) that land under
  `out3d_colab/` (already git-ignored).

**Quick start**

1. Run the **Setup** cell (Shift+Enter).
2. Edit the **Configuration** cell (preset / budget / backend / GPU).
3. Run **Launch locally** or **Launch on Colab**.
4. Run **Browse results** to inspect the latest run.


In [ ]:
# =============================================================================
# 0. Setup — locate the repo root and check the environment.
#    Run this cell first.
# =============================================================================
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

def _candidates():
    cwd = Path(os.getcwd()).resolve()
    yield cwd
    yield from cwd.parents
    # VS Code exposes the notebook path through this global (fallback only).
    nb = globals().get("__vsc_ipynb_file__")
    if nb:
        p = Path(nb).resolve().parent
        yield p
        yield from p.parents

REPO = None
for c in _candidates():
    if (c / "full3D_colab_run.py").exists():
        REPO = c
        break

if REPO is None:
    raise RuntimeError(
        "Could not find the repo root (looked for full3D_colab_run.py).\n"
        "Open this notebook inside TesisNov26/ or set REPO manually."
    )

os.chdir(REPO)
print(f"Repo root : {REPO}")

print(f"Python    : {sys.version.split()[0]}  ({sys.executable})")

for mod in ("numpy", "scipy"):
    try:
        m = __import__(mod)
        print(f"{mod:<9}: {m.__version__}")
    except ImportError:
        print(f"{mod:<9}: MISSING (required!)")

try:
    import torch
    print(f"{'torch':<9}: {torch.__version__}  cuda={torch.cuda.is_available()}")
except ImportError:
    print(f"{'torch':<9}: not installed (only needed for --backend torch)")

_colab = shutil.which("colab")
print(f"{'colab CLI':<9}: {_colab or 'not found (pip install google-colab-cli)'}")


## 1. Configuration

Edit the values below, then run the cell. The presets are documented in the
`full3D_colab_run.py` docstring:

| preset   | grid        | nsteps | M train/test | expected     |
|----------|-------------|--------|--------------|--------------|
| `smoke`  | 12x6x3      | 20     | 6/12         | ~seconds     |
| `quick`  | 20x10x5     | 40     | 12/24        | ~a minute    |
| `full`   | 24x12x6     | 80     | 24/48        | ~minutes     |
| `paper`  | 32x16x8     | 100    | 64/128       | ~tens of min |
| `paperGPU`| 64x32x16   | 100    | 64/128       | use torch GPU|

`BUDGET_MINUTES` is a hard wall-clock ceiling — the script calibrates one
evaluation and clamps the optimizer's evaluation count to fit inside it.


In [ ]:
# ============================ EDIT ME ======================================
PRESET         = "smoke"      # smoke | quick | full | paper | paperGPU
BUDGET_MINUTES = 5.0          # hard wall-clock budget for the optimizer
BACKEND        = "numpy"      # numpy (splu reference) | torch (batched GPU)
DEVICE         = "auto"       # auto | cuda | cpu | mps     (torch only)
DTYPE          = "float64"    # float64 | float32           (torch only)
ZIP_RESULTS    = True         # write colab3d_results.zip
PLOTS          = True         # write PNG figures
RUN_TAG        = "manual"     # short label for the output folder (no spaces)

# Optional grid/start overrides (None = use the preset's defaults).
NX       = None
NY       = None
NZ       = None
N_STARTS = None

# Optional fixed output dir (None = auto: out3d_colab/run_<PRESET>_<RUN_TAG>).
LOCAL_OUT = None
# ===========================================================================

# Colab launch defaults (used by the "Launch on Colab" cell below).
COLAB_SCRIPT = "full3D_colab_run.py"   # any repo script accepted by colab_runner.py
COLAB_GPU    = "T4"                     # T4 | L4 | A100 | V100 | CPU | none
COLAB_EXTRA  = f"--preset {PRESET} --budget {BUDGET_MINUTES} --backend {BACKEND} --zip"

def _out_dir():
    return LOCAL_OUT or f"out3d_colab/run_{PRESET}_{RUN_TAG}"

print("Configured run:")
print(f"  preset={PRESET}  budget={BUDGET_MINUTES}m  backend={BACKEND}  "
      f"device={DEVICE}  dtype={DTYPE}")
print(f"  zip={ZIP_RESULTS}  plots={PLOTS}  out={_out_dir()}")
print(f"  Colab: script={COLAB_SCRIPT}  gpu={COLAB_GPU}")


In [ ]:
# =============================================================================
# Helpers — build the command and stream subprocess output into the notebook.
# =============================================================================
def build_local_command():
    """Assemble the argv for full3D_colab_run.py from the config above."""
    cmd = [sys.executable, "-u", str(REPO / "full3D_colab_run.py")]
    cmd += ["--preset", PRESET, "--budget", str(BUDGET_MINUTES), "--backend", BACKEND]
    cmd += ["--device", DEVICE, "--dtype", DTYPE]
    if not PLOTS:
        cmd += ["--no-plots"]
    if ZIP_RESULTS:
        cmd += ["--zip"]
    if NX is not None:
        cmd += ["--nx", str(NX)]
    if NY is not None:
        cmd += ["--ny", str(NY)]
    if NZ is not None:
        cmd += ["--nz", str(NZ)]
    if N_STARTS is not None:
        cmd += ["--n-starts", str(N_STARTS)]
    cmd += ["--out", _out_dir()]
    return cmd


def _stream(cmd):
    print("$ " + " ".join(cmd))
    print("-" * 78)
    proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    print("-" * 78)
    print(f"Exit code: {proc.returncode}")
    return proc.returncode


def run_local():
    """Launch the configured simulation locally (streams output)."""
    return _stream(build_local_command())


## 2. Launch locally

Run this cell to execute `full3D_colab_run.py` on this machine. Output lands
in the folder printed above. Re-run after editing the Configuration cell.


In [ ]:
run_local()

## 3. Launch on Google Colab

This uses `colab_runner.py`, which wraps the `colab` CLI. It snapshots the
repo file tree before/after the run and reports exactly which files were
produced, plus a machine-readable manifest and the remote log.

**One-time prerequisites**

```bash
pip install google-colab-cli
gcloud auth application-default login      # ADC, no keyring needed
# or, if ADC is unavailable:
pip install keyring && colab auth && colab sessions --auth oauth2
```

Then run the next cell.


In [ ]:
def run_colab(script=COLAB_SCRIPT, gpu=COLAB_GPU, extra=COLAB_EXTRA):
    """Send `script` to a Colab runtime; args after `--` go to the script."""
    cmd = [sys.executable, "-u", str(REPO / "colab_runner.py"),
           script, "--gpu", gpu, "--"]
    cmd += extra.split()
    return _stream(cmd)


# For the GPU-sized preset (64x32x16, batched torch), the one-liner wrapper is
# easier — it provisions a GPU VM, runs, downloads results, and tears down:
#   python SCRIPTS/run_papergpu_colab.py --gpu A100 --budget 90
#   python SCRIPTS/run_papergpu_colab.py --dry-run

run_colab()

## 4. Browse results

Lists every run under `out3d_colab/`, then prints the `metrics.json` of the
most recent completed run and inlines its PNG figures.


In [ ]:
from IPython.display import Image, Markdown, display

def list_runs(pattern="out3d_colab/run_*"):
    dirs = sorted(REPO.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    return dirs


def show_results(pattern="out3d_colab/run_*", max_plots=4):
    dirs = list_runs(pattern)
    print(f"{len(dirs)} run folder(s) found\n")
    for d in dirs:
        has_metrics = (d / "metrics.json").exists()
        print(f"  {d.name:<30} metrics={'yes' if has_metrics else 'no ':>3}  "
              f"{time.ctime(d.stat().st_mtime)}")

    latest = None
    for d in dirs:
        if (d / "metrics.json").exists():
            latest = d
            break
    if latest is None:
        print("\nNo completed run (no metrics.json) yet.")
        return

    print("\n=== Latest completed run:", latest.name, "===")
    metrics = json.loads((latest / "metrics.json").read_text(encoding="utf-8"))
    print(json.dumps(metrics, indent=2))

    pngs = sorted(latest.glob("*.png"))
    if pngs:
        print(f"\n{len(pngs)} figure(s):")
        for p in pngs[:max_plots]:
            display(Markdown(f"**{p.name}**"))
            display(Image(filename=str(p)))
    else:
        print("\n(no PNG figures; was --no-plots set?)")

show_results()